# Озвучка русского текста (Silero v4)
Меню: Среда выполнения → Сменить тип → GPU (по желанию). Затем «Выполнить всё».
Текст обрабатывается на серверах Google Colab.

In [ ]:
#@title 1. Установка
!pip -q install torch numpy omegaconf
import torch, re, wave, subprocess, io
model, _ = torch.hub.load('snakers4/silero-models', 'silero_tts', language='ru', speaker='v4_ru', trust_repo=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print('Готово, устройство:', device)

In [ ]:
#@title 2. Выберите голос и загрузите текстовый файл .txt
speaker = 'xenia' #@param ['xenia','baya','aidar','eugene']
from google.colab import files
up = files.upload()
text = list(up.values())[0].decode('utf-8')
print('Знаков:', len(text))

In [ ]:
#@title 3. Озвучить и скачать
SR, MAX = 48000, 800
def chunks(t):
    buf = ''
    for s in re.split(r'(?<=[.!?…])\s+', t.replace('\n', ' ')):
        if buf and len(buf) + len(s) + 1 > MAX:
            yield buf; buf = ''
        buf = (buf + ' ' + s).strip()
    if buf: yield buf
pause = torch.zeros(int(SR * 0.3))
audio, parts = [], list(chunks(text))
for i, p in enumerate(parts, 1):
    audio += [model.apply_tts(text=p, speaker=speaker, sample_rate=SR, put_accent=True, put_yo=True).cpu(), pause]
    print(f'{i}/{len(parts)}', end=' ')
pcm = (torch.cat(audio).clamp(-1, 1) * 32767).short().numpy().tobytes()
with wave.open('out.wav', 'wb') as w:
    w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR); w.writeframes(pcm)
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', 'out.wav', '-b:a', '128k', 'voice.mp3'], check=True)
files.download('voice.mp3')